# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shaikh-Ashmaan/MachineLearning/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from google.colab import userdata
token = userdata.get("HF_TOKEN")
print("Token works:", bool(token))

Token works: True


## 1. Unit of analysis + time window

* **Chosen Lane:** Lane 2 — Refresh / Content Opportunity Scoring.
* **Unit of Analysis (Grain):** In `fact_content_daily_performance`, one row is `report_date + client_hash_id + content_hash_id`. For my Lane 2 scoring frame, I aggregate daily observations to **one row per `client_hash_id + content_hash_id`** at a fixed decision point (`2026-03-16`).
* **Tables & Time Window:** `fact_content_daily_performance` joined to `dim_content` and `dim_clients` on the mid-panel slice **`month=2026-03`** (`2026-03-01` to `2026-03-31`), using Days 1–15 for features and Days 16–31 for the target label (keeping `2026-06` sealed as a holdout test month).
* **Target / Proxy Predicted:** Binary upcoming visibility decline (`>20%` drop in second-half impressions vs. first-half impressions among pages with `>=100` prior impressions), used to rank a Top-50 editorial refresh review queue.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

* **Features (knowable before `2026-03-16`):**
  * `f1_prior_impressions`: Knowable at the decision moment because it sums GSC impressions strictly from `2026-03-01` to `2026-03-15`.
  * `f2_prior_ctr`: Knowable at the decision moment because it uses only closed prior-window clicks and impressions (`2026-03-01` to `2026-03-15`).
  * `f3_prior_avg_pos`: Knowable at the decision moment because it averages daily search positions recorded before `2026-03-16`.
  * `f4_prior_sessions`: Knowable at the decision moment because it sums GA4 sessions from `2026-03-01` to `2026-03-15` where `ga4_data_available IS TRUE`.
  * `f5_content_age_days`: Knowable at the decision moment because `content_created_at` from `dim_content` is a fixed historical timestamp measured up to `2026-03-16`.
* **Label / Target:** `target_decline` (`1` if `outcome_impressions` in `2026-03-16` to `2026-03-31` is `< 0.8 * f1_prior_impressions`, else `0`).
* **Context / Join Keys:** `client_hash_id`, `content_hash_id`, `report_date`, `ga4_data_available` (used for joins, client-grouped splits, and availability filtering—never as model features).
* **Excluded (and why):**
  * Post-decision metrics (`outcome_impressions`, future clicks/position) — excluded as features to prevent target leakage.
  * Rows where `ga4_data_available` is not `TRUE` — excluded so missing GA4 tracking before a client's `ga4_data_start` is never misread as zero traffic.
  * Any simulated product scores (`health_score`, `priority_score`) — excluded to avoid circular learning.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
import duckdb
import pandas as pd
from google.colab import userdata
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

hf_token = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}');")

REL = "hf://datasets/FlyRank/internship-warehouse"
SLICE_PATH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"

# Auto-detect metric column names
cols = set(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{SLICE_PATH}')").df()["column_name"])
impr_col = "gsc_impressions" if "gsc_impressions" in cols else "impressions"
click_col = "gsc_clicks" if "gsc_clicks" in cols else "clicks"
pos_col = "gsc_avg_position" if "gsc_avg_position" in cols else "avg_position"
sess_col = "ga4_sessions" if "ga4_sessions" in cols else "sessions"

# QUERY 1: Prove the grain
q1 = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS unique_grain_keys
    FROM read_parquet('{SLICE_PATH}')
""").df()
print("=== QUERY 1: Grain Verification ===")
print(q1.to_string(index=False))

# QUERY 2: Row count & exact date span
q2 = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT client_hash_id) AS distinct_clients,
           COUNT(DISTINCT content_hash_id) AS distinct_content_items,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM read_parquet('{SLICE_PATH}')
""").df()
print("\n=== QUERY 2: Slice Row Count & Date Span ===")
print(q2.to_string(index=False))

# QUERY 3: Availability check using IS TRUE
q3 = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS rows_with_ga4_true,
           ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 2) AS pct_ga4_available
    FROM read_parquet('{SLICE_PATH}')
""").df()
print("\n=== QUERY 3: Availability Check (IS TRUE) ===")
print(q3.to_string(index=False))

# Build 5-feature frame (Days 1-15 features -> Days 16-31 target)
feat_df = con.sql(f"""
    WITH daily AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,
            f.report_date,
            COALESCE(f.{impr_col}, 0) AS impr,
            COALESCE(f.{click_col}, 0) AS clicks,
            f.{pos_col} AS pos,
            COALESCE(f.{sess_col}, 0) AS sessions,
            DATEDIFF('day', c.content_created_date, DATE '2026-03-16') AS content_age_days
        FROM read_parquet('{SLICE_PATH}') f
        LEFT JOIN read_parquet('{REL}/dim_content.parquet') c
            ON f.content_hash_id = c.content_hash_id
        WHERE f.ga4_data_available IS TRUE
    ),
    agg AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN impr ELSE 0 END) AS f1_prior_impressions,
            SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN clicks ELSE 0 END) * 1.0
                / NULLIF(SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN impr ELSE 0 END), 0) AS f2_prior_ctr,
            AVG(CASE WHEN report_date <= DATE '2026-03-15' THEN pos END) AS f3_prior_avg_pos,
            SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN sessions ELSE 0 END) AS f4_prior_sessions,
            MAX(content_age_days) AS f5_content_age_days,
            SUM(CASE WHEN report_date >= DATE '2026-03-16' THEN impr ELSE 0 END) AS outcome_impressions
        FROM daily
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT *, CASE WHEN outcome_impressions < 0.8 * f1_prior_impressions THEN 1 ELSE 0 END AS target_decline
    FROM agg
    WHERE f1_prior_impressions >= 100 AND f5_content_age_days >= 30
""").df().fillna(0)

feature_cols = ["f1_prior_impressions", "f2_prior_ctr", "f3_prior_avg_pos", "f4_prior_sessions", "f5_content_age_days"]
print(f"\n=== 5-FEATURE FRAME ({feat_df.shape[0]:,} items | Decline rate: {feat_df['target_decline'].mean():.1%}) ===")
print(feat_df[["client_hash_id", "content_hash_id"] + feature_cols + ["target_decline"]].head(5).to_string(index=False))

# Deliberate leakage experiment
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(feat_df, feat_df["target_decline"], groups=feat_df["client_hash_id"]))

clf_honest = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
clf_honest.fit(feat_df.iloc[train_idx][feature_cols], feat_df.iloc[train_idx]["target_decline"])
honest_auc = roc_auc_score(feat_df.iloc[test_idx]["target_decline"], clf_honest.predict_proba(feat_df.iloc[test_idx][feature_cols])[:, 1])

feat_df["LEAKY_future_to_prior_ratio"] = feat_df["outcome_impressions"] / (feat_df["f1_prior_impressions"] + 1)
clf_leaky = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
clf_leaky.fit(feat_df.iloc[train_idx][feature_cols + ["LEAKY_future_to_prior_ratio"]], feat_df.iloc[train_idx]["target_decline"])
leaky_auc = roc_auc_score(feat_df.iloc[test_idx]["target_decline"], clf_leaky.predict_proba(feat_df.iloc[test_idx][feature_cols + ["LEAKY_future_to_prior_ratio"]])[:, 1])

feat_df.drop(columns=["LEAKY_future_to_prior_ratio", "outcome_impressions"], inplace=True)

print(f"\n=== LEAKAGE EXPERIMENT ===")
print(f"Honest ROC AUC (5 knowable features): {honest_auc:.3f}")
print(f"Leaky ROC AUC (with label-derived feature): {leaky_auc:.3f}")
print("Leaky column deleted. Clean columns:", list(feat_df.columns))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== QUERY 1: Grain Verification ===
 total_rows  unique_grain_keys
    9841378            9841378

=== QUERY 2: Slice Row Count & Date Span ===
 total_rows  distinct_clients  distinct_content_items   min_date   max_date
    9841378                55                  331437 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== QUERY 3: Availability Check (IS TRUE) ===
 total_rows  rows_with_ga4_true  pct_ga4_available
    9841378              413966               4.21


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== 5-FEATURE FRAME (15,852 items | Decline rate: 32.7%) ===
         client_hash_id          content_hash_id  f1_prior_impressions  f2_prior_ctr  f3_prior_avg_pos  f4_prior_sessions  f5_content_age_days  target_decline
client_e547b89c05043229 content_a6eb550e132505fd                7716.0      0.009850          4.875942               82.0                  336               0
client_e547b89c05043229 content_cbb44b4a10088d7b                3258.0      0.008287          2.895804               33.0                  336               0
client_e547b89c05043229 content_1a8a9b1a576ae25a                 227.0      0.013216          3.353216                4.0                  336               0
client_e547b89c05043229 content_2f1604b73f465db2                 218.0      0.013761          2.947463                4.0                  336               1
client_e547b89c05043229 content_5538d1fc8d19fa4a                8994.0      0.005337          3.469778               61.0                  336 

## 4. Data limits

### Named Limitation of This Slice
* **Short-Window Noise, Unbalanced Panel & Seasonality:** A single 31-day slice (`month=2026-03`, split 15 days vs. 16 days) cannot separate true structural content decay from short-term weekly fluctuations, calendar seasonality, or internal URL consolidation across sibling pages. Additionally, different clients have different `gsc_data_start` and `ga4_data_start` dates (early rows can be GSC-only), and observational warehouse logs cannot prove that an editorial refresh *caused* recovery.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.